# Supermarket Sales Analysis
### IBM SkillsBuild / AICTE — Data Analytics Project

**Student:** Affaan  
**Project:** Supermarket Sales Analysis  
**Environment:** Google Colab / Jupyter Notebook

## Objective

Analyze supermarket sales data to understand product, category, branch, customer, payment and rating performance, then convert the analysis into business insights and decisions.

**Workflow:** Load → Clean → Validate → Analyze → Visualize → Interpret → Recommend


## 1. Problem Statement

The aim of this project is to analyze supermarket sales data and find useful information about products, branches, categories, customers, payments, and ratings.

The supplied project dataset contains 500 sales transactions with fields covering product, branch, city, customer type, quantity, price, payment method, rating and sales.

The required sales relationship is:

**Sales = Quantity × Unit Price**


## 2. Required Analysis

1. Collect and load the dataset.
2. Check the data for missing or incorrect values.
3. Calculate and validate `Sales = Quantity × Unit Price`.
4. Group and summarize the data using totals, counts and averages.
5. Create charts to compare the results.
6. Use the results to make business decisions.

The notebook also documents five visualizations, five observations, five insights, three hypotheses and three recommendations.


## 3. Import Libraries

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")


## 4. Load Dataset

### Google Colab
Run the next cell and upload the supermarket dataset.

The loader supports CSV and Excel files so that the notebook can be used with the supplied project dataset.


In [ ]:
try:
    from google.colab import files
    print("Upload the supermarket dataset (.csv, .xlsx or .xls):")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    DATA_FILE = next(iter(uploaded.keys()))
except ImportError:
    candidates = glob.glob("*.csv") + glob.glob("*.xlsx") + glob.glob("*.xls")
    if not candidates:
        raise FileNotFoundError(
            "No dataset found. Put the project dataset in the same folder as this notebook."
        )
    DATA_FILE = candidates[0]

print("Selected dataset:", DATA_FILE)


In [ ]:
ext = os.path.splitext(DATA_FILE)[1].lower()

if ext == ".csv":
    df = pd.read_csv(DATA_FILE)
elif ext in [".xlsx", ".xls"]:
    df = pd.read_excel(DATA_FILE)
else:
    raise ValueError("Unsupported file type.")

print("Dataset loaded.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())


## 5. Dataset Structure

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDataset information:")
df.info()


## 6. Data Quality — Missing Values

In [ ]:
missing_report = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})

display(missing_report)
print("Total missing cells:", int(df.isna().sum().sum()))


## 7. Data Quality — Duplicate Records

In [ ]:
duplicate_count = int(df.duplicated().sum())
print("Duplicate rows:", duplicate_count)

if duplicate_count:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")


## 8. Standardize Required Column References

In [ ]:
def norm(x):
    return str(x).strip().lower().replace("_", " ").replace("-", " ")

normalized = {norm(c): c for c in df.columns}

def find_col(options, required=True):
    for option in options:
        if norm(option) in normalized:
            return normalized[norm(option)]
    if required:
        raise KeyError(f"Required column not found: {options}. Available: {list(df.columns)}")
    return None

COL = {
    "invoice": find_col(["Invoice ID", "Invoice_ID", "Invoice"]),
    "date": find_col(["Date", "Order Date", "Order_Date"]),
    "branch": find_col(["Branch"]),
    "city": find_col(["City"]),
    "customer_type": find_col(["Customer Type", "Customer_Type"]),
    "gender": find_col(["Gender"]),
    "product": find_col(["Product"]),
    "category": find_col(["Category"]),
    "quantity": find_col(["Quantity", "Qty"]),
    "unit_price": find_col(["Unit Price", "Unit_Price", "Price"]),
    "payment": find_col(["Payment", "Payment Method", "Payment_Method"]),
    "rating": find_col(["Rating", "Customer Rating", "Customer_Rating"]),
    "sales": find_col(["Sales", "Revenue", "Total Sales", "Total_Sales"])
}

display(pd.DataFrame({"Logical Field": COL.keys(), "Dataset Column": COL.values()}))


## 9. Data Type Conversion and Validation

In [ ]:
df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")

for key in ["quantity", "unit_price", "rating", "sales"]:
    df[COL[key]] = pd.to_numeric(df[COL[key]], errors="coerce")

print("Invalid dates:", int(df[COL["date"]].isna().sum()))
for key in ["quantity", "unit_price", "rating", "sales"]:
    print(f"Invalid numeric values in {COL[key]}:", int(df[COL[key]].isna().sum()))


## 10. Missing-Value Treatment

In [ ]:
# Create an analysis copy.
clean_df = df.copy()

# Numeric columns: median treatment where necessary.
for key in ["quantity", "unit_price", "rating", "sales"]:
    col = COL[key]
    if clean_df[col].isna().any():
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# Categorical columns: mode treatment where necessary.
for key in ["branch", "city", "customer_type", "gender", "product", "category", "payment"]:
    col = COL[key]
    if clean_df[col].isna().any():
        mode = clean_df[col].mode(dropna=True)
        if not mode.empty:
            clean_df[col] = clean_df[col].fillna(mode.iloc[0])

print("Remaining missing cells:", int(clean_df.isna().sum().sum()))


## 11. Required Formula Validation

The project requires:

**Sales = Quantity × Unit Price**

The following cell verifies the relationship for every transaction.


In [ ]:
clean_df["Calculated_Sales"] = clean_df[COL["quantity"]] * clean_df[COL["unit_price"]]
clean_df["Sales_Difference"] = clean_df[COL["sales"]] - clean_df["Calculated_Sales"]

matches = np.isclose(
    clean_df[COL["sales"]],
    clean_df["Calculated_Sales"],
    rtol=1e-9,
    atol=1e-6
)

print("Records checked:", len(clean_df))
print("Matching records:", int(matches.sum()))
print("Non-matching records:", int((~matches).sum()))
print("Maximum absolute difference:", float(clean_df["Sales_Difference"].abs().max()))

if matches.all():
    print("PASS — Sales matches Quantity × Unit Price for all records.")
else:
    print("WARNING — Some records do not match the required formula.")
    display(clean_df.loc[~matches, [
        COL["invoice"], COL["quantity"], COL["unit_price"],
        COL["sales"], "Calculated_Sales", "Sales_Difference"
    ]].head(20))


## 12. Descriptive Statistics

In [ ]:
numeric_cols = [COL["quantity"], COL["unit_price"], COL["sales"], COL["rating"]]
display(clean_df[numeric_cols].describe().T)

summary = pd.DataFrame({
    "Mean": clean_df[numeric_cols].mean(),
    "Median": clean_df[numeric_cols].median(),
    "Minimum": clean_df[numeric_cols].min(),
    "Maximum": clean_df[numeric_cols].max(),
    "Std Dev": clean_df[numeric_cols].std()
})
display(summary.round(2))


## 13. Outlier Analysis

In [ ]:
outliers = []

for col in numeric_cols:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    count = int(((clean_df[col] < low) | (clean_df[col] > high)).sum())

    outliers.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": low,
        "Upper Bound": high,
        "Outlier Count": count
    })

display(pd.DataFrame(outliers).round(2))


# 14. Five Required Visualizations

## Visualization 1 — Monthly Sales Trend
**Question:** Which months generated the highest and lowest sales?


In [ ]:
time_df = clean_df.dropna(subset=[COL["date"]]).copy()
time_df["Month"] = time_df[COL["date"]].dt.to_period("M")

monthly_sales = time_df.groupby("Month")[COL["sales"]].sum().sort_index()

plt.figure(figsize=(11, 5))
plt.plot(monthly_sales.index.astype(str), monthly_sales.values, marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

highest_month = monthly_sales.idxmax()
lowest_month = monthly_sales.idxmin()

print(f"Observation: Highest month = {highest_month}, ₹{monthly_sales.max():,.2f}")
print(f"Observation: Lowest month = {lowest_month}, ₹{monthly_sales.min():,.2f}")


## Visualization 2 — Category Performance
**Question:** Which category sells the most?


In [ ]:
category_sales = clean_df.groupby(COL["category"])[COL["sales"]].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
category_sales.sort_values().plot(kind="barh")
plt.title("Total Sales by Product Category")
plt.xlabel("Sales")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

best_category = category_sales.idxmax()
print(f"Observation: {best_category} has the highest category sales: ₹{category_sales.max():,.2f}")


## Visualization 3 — Customer Type
**Question:** How are transactions distributed between Member and Normal customers?


In [ ]:
customer_counts = clean_df[COL["customer_type"]].value_counts()

plt.figure(figsize=(8, 5))
customer_counts.plot(kind="bar")
plt.title("Transactions by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

largest_customer_type = customer_counts.idxmax()
print(f"Observation: {largest_customer_type} has the most transactions: {customer_counts.max()}")


## Visualization 4 — Branch Performance
**Question:** Which branch performs best?


In [ ]:
branch_sales = clean_df.groupby(COL["branch"])[COL["sales"]].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
branch_sales.sort_values().plot(kind="barh")
plt.title("Total Sales by Branch")
plt.xlabel("Sales")
plt.ylabel("Branch")
plt.tight_layout()
plt.show()

best_branch = branch_sales.idxmax()
best_branch_sales = branch_sales.max()
best_branch_city = clean_df.loc[
    clean_df[COL["branch"]] == best_branch, COL["city"]
].mode().iloc[0]

print(f"Observation: Branch {best_branch} ({best_branch_city}) has the highest sales: ₹{best_branch_sales:,.2f}")


## Visualization 5 — Top Products
**Question:** Which products generate the highest sales?


In [ ]:
product_sales = clean_df.groupby(COL["product"])[COL["sales"]].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
product_sales.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Products by Sales")
plt.xlabel("Sales")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

best_product = product_sales.idxmax()
best_product_sales = product_sales.max()

print(f"Observation: {best_product} has the highest product sales: ₹{best_product_sales:,.2f}")


## 15. Payment Method Analysis

In [ ]:
payment_counts = clean_df[COL["payment"]].value_counts()

display(payment_counts.to_frame("Transactions"))

most_used_payment = payment_counts.idxmax()
most_used_payment_count = int(payment_counts.max())

plt.figure(figsize=(8, 5))
payment_counts.plot(kind="bar")
plt.title("Payment Method Usage")
plt.xlabel("Payment Method")
plt.ylabel("Transactions")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f"Most popular payment method: {most_used_payment} — {most_used_payment_count} transactions")


## 16. Member vs Normal Customer Spending

In [ ]:
customer_spending = clean_df.groupby(COL["customer_type"])[COL["sales"]].agg(
    Transactions="count",
    Total_Sales="sum",
    Average_Transaction="mean"
)

display(customer_spending.round(2))

member_label = next((x for x in customer_spending.index if str(x).lower() == "member"), None)
normal_label = next((x for x in customer_spending.index if str(x).lower() == "normal"), None)

if member_label and normal_label:
    member_avg = customer_spending.loc[member_label, "Average_Transaction"]
    normal_avg = customer_spending.loc[normal_label, "Average_Transaction"]

    print(f"Member average transaction: ₹{member_avg:,.2f}")
    print(f"Normal average transaction: ₹{normal_avg:,.2f}")

    if member_avg > normal_avg:
        print("Result: Members have the higher average transaction value.")
    elif member_avg < normal_avg:
        print("Result: Normal customers have the higher average transaction value.")
    else:
        print("Result: Both groups have the same average transaction value.")


## 17. Customer Rating Analysis

In [ ]:
average_rating = clean_df[COL["rating"]].mean()
rating_by_branch = clean_df.groupby(COL["branch"])[COL["rating"]].mean().sort_values(ascending=False)

print(f"Average customer rating: {average_rating:.2f} / 5")
display(rating_by_branch.to_frame("Average Rating").round(2))

plt.figure(figsize=(9, 5))
rating_by_branch.plot(kind="bar")
plt.title("Average Customer Rating by Branch")
plt.xlabel("Branch")
plt.ylabel("Average Rating")
plt.ylim(0, 5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 18. Correlation Analysis

In [ ]:
corr = clean_df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

display(corr.round(2))


# 19. Five Observations

Observations state what is directly visible in the data.


In [ ]:
observations = [
    f"1. {highest_month} recorded the highest monthly sales at ₹{monthly_sales.max():,.2f}.",
    f"2. {best_category} recorded the highest category sales at ₹{category_sales.max():,.2f}.",
    f"3. {largest_customer_type} customers account for the largest number of transactions ({customer_counts.max()}).",
    f"4. Branch {best_branch} ({best_branch_city}) recorded the highest branch sales at ₹{best_branch_sales:,.2f}.",
    f"5. {best_product} recorded the highest product sales at ₹{best_product_sales:,.2f}."
]

for x in observations:
    print(x)


# 20. Five Data-Supported Insights

Insights explain why the observed patterns matter. They do not invent causes that the data cannot prove.


In [ ]:
total_sales = clean_df[COL["sales"]].sum()
top_product_share = best_product_sales / total_sales * 100
top_category_share = category_sales.max() / total_sales * 100
top_branch_share = best_branch_sales / total_sales * 100

insights = [
    f"1. {best_product} contributes {top_product_share:.2f}% of total sales, so high-sales products deserve close inventory monitoring.",
    f"2. {best_category} contributes {top_category_share:.2f}% of total sales, making category-level planning important.",
    f"3. Branch {best_branch} contributes {top_branch_share:.2f}% of total sales, providing a useful benchmark for branch comparison.",
    f"4. {most_used_payment} is the most-used payment method with {most_used_payment_count} transactions, so payment-channel usage should be considered operationally.",
    f"5. The average rating is {average_rating:.2f}/5, so customer experience can be monitored alongside sales rather than using revenue alone."
]

for x in insights:
    print(x)


# 21. Three Hypotheses

A hypothesis is a possible explanation, not a confirmed fact.

The dataset does not contain direct information about promotions, stockouts, staffing,
local competition or customer motivation, so these explanations are intentionally labelled as hypotheses.


In [ ]:
hypotheses = [
    f"Hypothesis 1 — Inventory or demand: The strong sales of {best_product} may be related to sustained demand or effective product availability.",
    f"Hypothesis 2 — Branch conditions: Branch {best_branch} may have stronger local demand, product mix, pricing or operating conditions contributing to its higher sales.",
    f"Hypothesis 3 — Category mix: The strong performance of {best_category} could be related to its product mix, purchase frequency or transaction quantities."
]

for x in hypotheses:
    print(x)


# 22. Three Business Recommendations

Recommendations are based on the observed data and are written as practical actions.


In [ ]:
recommendations = [
    f"1. Inventory planning — Closely monitor stock availability for {best_product} and other high-sales products, and review category-level demand before replenishment.",
    f"2. Branch benchmarking — Compare Branch {best_branch}'s product mix, customer mix and transaction patterns with other branches to identify practices worth investigating.",
    f"3. Customer/payment operations — Continue supporting {most_used_payment}, while monitoring payment usage and branch-level ratings for service improvement."
]

for x in recommendations:
    print(x)


## 23. Live Analysis Answers

In [ ]:
print("PROJECT ANSWERS")
print("=" * 70)

print(f"1. Highest-sales product: {best_product} — ₹{best_product_sales:,.2f}")
print(f"2. Best-performing branch: Branch {best_branch} ({best_branch_city}) — ₹{best_branch_sales:,.2f}")
print(f"3. Highest-sales category: {best_category} — ₹{category_sales.max():,.2f}")
print(f"4. Most-used payment method: {most_used_payment} — {most_used_payment_count} transactions")

if member_label and normal_label:
    print(f"5. Member average: ₹{member_avg:,.2f}")
    print(f"   Normal average: ₹{normal_avg:,.2f}")

print(f"6. Average customer rating: {average_rating:.2f} / 5")


## 24. Project-Document Reference Validation

The supplied project document records the following reference results:

- Cheese — ₹27,906.30 highest product sales
- Branch C (Mumbai) — ₹72,469.45 highest branch sales
- Beverages — ₹56,108.24 highest category sales
- UPI — 127 transactions
- Member average — ₹483.14
- Normal average — ₹497.07
- Average rating — 3.99 / 5

The next cell compares the calculated values with those references.


In [ ]:
checks = [
    ("Highest product", f"{best_product} — ₹{best_product_sales:,.2f}",
     "Cheese — ₹27,906.30",
     best_product == "Cheese" and np.isclose(best_product_sales, 27906.30, atol=0.01)),

    ("Best branch", f"Branch {best_branch} ({best_branch_city}) — ₹{best_branch_sales:,.2f}",
     "Branch C (Mumbai) — ₹72,469.45",
     best_branch == "C" and np.isclose(best_branch_sales, 72469.45, atol=0.01)),

    ("Highest category", f"{best_category} — ₹{category_sales.max():,.2f}",
     "Beverages — ₹56,108.24",
     best_category == "Beverages" and np.isclose(category_sales.max(), 56108.24, atol=0.01)),

    ("Most-used payment", f"{most_used_payment} — {most_used_payment_count}",
     "UPI — 127",
     most_used_payment == "UPI" and most_used_payment_count == 127),

    ("Average rating", f"{average_rating:.2f}",
     "3.99",
     np.isclose(average_rating, 3.99, atol=0.01))
]

if member_label and normal_label:
    checks.extend([
        ("Member average", f"₹{member_avg:.2f}", "₹483.14", np.isclose(member_avg, 483.14, atol=0.01)),
        ("Normal average", f"₹{normal_avg:.2f}", "₹497.07", np.isclose(normal_avg, 497.07, atol=0.01))
    ])

validation = pd.DataFrame(checks, columns=["Metric", "Calculated", "Project Reference", "Matches"])
display(validation)

print("Reference validation:",
      "ALL MATCH" if validation["Matches"].all()
      else "ONE OR MORE DIFFERENCES — verify the uploaded dataset.")


## 25. Export Project Outputs

In [ ]:
export_df = clean_df.drop(columns=["Calculated_Sales", "Sales_Difference"], errors="ignore")

export_df.to_csv("supermarket_sales_cleaned.csv", index=False)
category_sales.to_csv("category_sales_summary.csv", header=["Total Sales"])
branch_sales.to_csv("branch_sales_summary.csv", header=["Total Sales"])
product_sales.to_csv("product_sales_summary.csv", header=["Total Sales"])
payment_counts.to_csv("payment_method_summary.csv", header=["Transactions"])
monthly_sales.to_csv("monthly_sales_summary.csv", header=["Total Sales"])

kpi = pd.DataFrame({
    "Metric": [
        "Total Transactions", "Total Sales", "Total Quantity",
        "Average Transaction", "Average Rating",
        "Highest Product", "Highest Category",
        "Best Branch", "Most Used Payment"
    ],
    "Value": [
        len(clean_df), clean_df[COL["sales"]].sum(), clean_df[COL["quantity"]].sum(),
        clean_df[COL["sales"]].mean(), average_rating,
        best_product, best_category, f"Branch {best_branch}", most_used_payment
    ]
})
kpi.to_csv("supermarket_kpi_summary.csv", index=False)

print("Exported cleaned dataset and summary files successfully.")


# 26. Conclusion

This project demonstrates how supermarket transaction data can be converted into useful business information through data cleaning, formula validation, aggregation, visualization and interpretation.

The analysis covers products, categories, branches, customers, payment methods and ratings. The results are used to support decisions about inventory, branch benchmarking, payment channels, customer experience and membership planning.

The project follows the required analytical progression:

**Observation → Insight → Hypothesis → Recommendation**

Observations describe the data, insights explain why the patterns matter, hypotheses provide possible explanations without claiming proof, and recommendations translate the evidence into practical actions.


## 27. Final Submission Checklist

- [x] Complete `.ipynb` code
- [x] `requirements.txt`
- [x] Project report
- [x] `README.md`
- [x] Dataset loading
- [x] Data quality checks
- [x] Sales formula validation
- [x] Aggregation and descriptive analysis
- [x] Required charts
- [x] Five observations
- [x] Five insights
- [x] Three hypotheses
- [x] Three recommendations
- [x] Business decisions
- [x] Conclusion
